# Ingest Sprints.json files
1. Read the files using spark dataframe reader API
2. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, FloatType
sprints_schema = StructType([
    StructField("date", DateType()),
    StructField("raceName", StringType()) ,
    StructField("round", IntegerType()),
    StructField("season", IntegerType()),
    StructField("url", StringType()),
    StructField("constructorId", StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", FloatType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType())
])

In [0]:
sprints_df = (
    spark.read
    .format("json")
    .option('header', 'true')
    .option('multiLine', True)   
    .schema(sprints_schema)
    .load(source_file)
)

In [0]:
display(sprints_df)

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

In [0]:
display(sprints_final_df)


In [0]:
(
    sprints_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
)


In [0]:
display(spark.read.table(table_name))